# Exploratory Data Analysis — Diabetes Prediction

This notebook explores `data/diabetes.csv` before training. It mirrors the
cleaning logic in `model/train_model.py` and the plots in
`model/evaluate_model.py`, but lets you interactively inspect intermediate
results.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

df = pd.read_csv("../data/diabetes.csv")
df.head()

In [ ]:
df.info()
df.describe()

## Missing values (encoded as 0 for certain columns)

In this dataset, `Glucose`, `BloodPressure`, `SkinThickness`, `Insulin`, and
`BMI` cannot realistically be zero for a living patient — a recorded 0 in
these columns represents a missing measurement.

In [ ]:
zero_as_missing = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]
for col in zero_as_missing:
    print(col, "-> zeros:", (df[col] == 0).sum())

In [ ]:
df_clean = df.drop_duplicates().copy()
for col in zero_as_missing:
    df_clean[col] = df_clean[col].replace(0, np.nan)
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())

for col in df_clean.columns.drop("Outcome"):
    lower, upper = df_clean[col].quantile(0.01), df_clean[col].quantile(0.99)
    df_clean[col] = df_clean[col].clip(lower, upper)

df_clean.describe()

## Class balance

In [ ]:
sns.countplot(x="Outcome", data=df_clean, palette=["#22c55e", "#ef4444"])
plt.title("Class Balance")
plt.show()
print(df_clean["Outcome"].value_counts(normalize=True))

**Observation:** the dataset is moderately imbalanced (roughly 65% non-diabetic vs. 35% diabetic), which is why the training pipeline selects the best model by F1 score rather than raw accuracy.

## Feature distributions

In [ ]:
features = [c for c in df_clean.columns if c != "Outcome"]
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, col in zip(axes.flatten(), features):
    sns.histplot(df_clean[col], kde=True, ax=ax, color="#3b82f6")
    ax.set_title(col)
plt.tight_layout()
plt.show()

## Correlation with Outcome

In [ ]:
plt.figure(figsize=(9, 7))
corr = df_clean.corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation Matrix")
plt.show()
print(corr["Outcome"].sort_values(ascending=False))

**Observation:** `Glucose` shows the strongest correlation with `Outcome`, followed by `BMI` and `Age` — consistent with established clinical risk factors for type 2 diabetes.

## Feature spread by outcome

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, col in zip(axes.flatten(), features):
    sns.boxplot(x="Outcome", y=col, data=df_clean, ax=ax,
                palette=["#22c55e", "#ef4444"])
    ax.set_title(col)
plt.tight_layout()
plt.show()

**Observation:** `Glucose` and `BMI` show the clearest separation in median values between diabetic and non-diabetic patients, matching the correlation results above.

See `model/train_model.py` and `model/evaluate_model.py` for the full training pipeline and additional evaluation plots (ROC curve, confusion matrix, feature importance).